# CoRe-TFM — JMLR Robustness EXECUTABLE V2

This notebook executes the fresh robustness experiments rather than merely planning them. It reuses the original `CoRe_TFM_Q1_FAST_COMPLETE_256_Colab.ipynb` as the inference engine and patches only pre-frozen protocol constants.

### Resumability
Each run writes checkpoints to Google Drive. Re-running this notebook selects the **next incomplete variant**. If Colab disconnects, reopen this notebook and run all again; completed fold tasks are not repeated.

### Evidence guardrail
`COMPLETE.json` is written only after the expected fold matrix exists. The original frozen Q1 evidence directory is never modified.


## 1. Setup — always pull latest `main`

In [ ]:
from pathlib import Path
import json, os, subprocess, sys, shutil
import pandas as pd
import numpy as np

from google.colab import drive
drive.mount('/content/drive')

ROOT = Path('/content/core-tfm')
if not (ROOT/'.git').exists():
    subprocess.run(['git','clone','https://github.com/bnssaanirudh/core-tfm.git',str(ROOT)], check=True)
subprocess.run(['git','fetch','origin'], cwd=ROOT, check=True)
subprocess.run(['git','checkout','main'], cwd=ROOT, check=True)
subprocess.run(['git','reset','--hard','origin/main'], cwd=ROOT, check=True)
HEAD = subprocess.check_output(['git','rev-parse','HEAD'], cwd=ROOT, text=True).strip()
print('HEAD:', HEAD)

subprocess.run([sys.executable,'-m','pip','install','-q','-e','.[test]','pyyaml','ucimlrepo','hf_transfer'], cwd=ROOT, check=True)
sys.path.insert(0, str(ROOT/'src'))
os.chdir(ROOT)

from core_tfm.robustness_runner import load_notebook, patch_q1_notebook, code_cells_through_shard_12e, fold_result_status, write_complete_marker

ROBUST_ROOT = Path('/content/drive/MyDrive/CoRe_TFM_Q1/core_tfm_jmlr_robustness_v2')
ROBUST_ROOT.mkdir(parents=True, exist_ok=True)
print('Robustness root:', ROBUST_ROOT)


## 2. Freeze/check the robustness protocol

In [ ]:
import yaml
cfg = yaml.safe_load((ROOT/'configs'/'reliability_aware_experiments.yaml').read_text())
PROTOCOL_PATH = ROBUST_ROOT/'ROBUSTNESS_PROTOCOL.json'
protocol = {
    'run_id': 'core_tfm_jmlr_robustness_v2',
    'source_commit_at_freeze': HEAD,
    'inference_engine': 'notebooks/CoRe_TFM_Q1_FAST_COMPLETE_256_Colab.ipynb',
    'multi_seed': cfg['seed_robustness'],
    'context_size': cfg['context_size'],
    'safe_selective': cfg['safe_selective'],
    'rare_class_sensitivity': cfg['rare_class_sensitivity'],
    'primary_models': cfg['models']['primary'],
    'outcome_blind_freeze': True,
}
if PROTOCOL_PATH.exists():
    frozen_protocol = json.loads(PROTOCOL_PATH.read_text())
    print('Using existing frozen protocol.')
else:
    PROTOCOL_PATH.write_text(json.dumps(protocol, indent=2))
    frozen_protocol = protocol
    print('Protocol frozen now.')
display(frozen_protocol)


## 3. Choose the next incomplete real-TFM variant

Priority is multi-seed first, then context-size. One variant can itself resume internally because the original Q1 notebook checkpoints completed fold tasks.


In [ ]:
seed_variants = [
    {
        'group':'multi_seed', 'seed':int(seed), 'train_limit':int(cfg['seed_robustness']['train_limit']),
        'test_limit':int(cfg['seed_robustness']['test_limit']), 'name':f'seed_{seed}'
    }
    for seed in cfg['seed_robustness']['seeds']
]
context_variants = [
    {
        'group':'context_size', 'seed':int(seed), 'train_limit':int(size), 'test_limit':128,
        'name':f'seed_{seed}_train_{size}'
    }
    for seed in cfg['context_size']['seeds']
    for size in cfg['context_size']['train_sizes']
]
QUEUE = seed_variants + context_variants

status_rows=[]
NEXT=None
for v in QUEUE:
    run_dir = ROBUST_ROOT/v['group']/v['name']
    st = fold_result_status(run_dir)
    status_rows.append({**v, **{k:st.get(k) for k in ['complete','rows','fold_cells']}})
    if NEXT is None and not st.get('complete', False):
        NEXT = v
display(pd.DataFrame(status_rows))
print('NEXT:', NEXT)


## 4. Execute/resume the next variant using the original Q1 engine

This is the expensive GPU cell. It executes the original setup/data/model/fold/shard code through shard 12E. The model adapters, data preparation, conditional-query logic, candidate construction, validation selection, and scoring remain the original Q1 implementation.


In [ ]:
if NEXT is None:
    print('All multi-seed and context-size variants are structurally complete.')
else:
    template = load_notebook(ROOT/'notebooks'/'CoRe_TFM_Q1_FAST_COMPLETE_256_Colab.ipynb')
    patched = patch_q1_notebook(
        template,
        run_id=f"{NEXT['group']}/{NEXT['name']}",
        seed=NEXT['seed'],
        train_limit=NEXT['train_limit'],
        test_limit=NEXT['test_limit'],
        drive_base=str(ROBUST_ROOT),
        session_minutes=600,
        disable_controlled_replications=True,
        disable_selection_ablations=True,
        disable_validation_sensitivity=True,
    )
    sources = code_cells_through_shard_12e(patched)
    print(f"Executing {len(sources)} original Q1 code cells for {NEXT}")
    engine_ns = {'__name__':'__robustness_exec__'}
    for idx, source in enumerate(sources, 1):
        print(f'--- Q1 engine cell {idx}/{len(sources)} ---', flush=True)
        exec(compile(source, f'<q1_engine_cell_{idx}>', 'exec'), engine_ns, engine_ns)

    run_dir = ROBUST_ROOT/NEXT['group']/NEXT['name']
    st = fold_result_status(run_dir)
    print('Variant status:', json.dumps(st, indent=2))
    if st.get('complete'):
        marker = write_complete_marker(run_dir, {
            'group':NEXT['group'], 'seed':NEXT['seed'],
            'requested_train_limit':NEXT['train_limit'], 'source_commit':HEAD,
            'inference_engine':'original_Q1_notebook_through_12E',
        })
        print('COMPLETE:', marker)
    else:
        print('Variant remains incomplete. Re-run this notebook; it will resume the same variant.')


## 5. Aggregate real-TFM robustness progress and rare-class exclusion sensitivity

In [ ]:
subprocess.run([
    sys.executable, str(ROOT/'experiments'/'summarize_robustness_runs.py'),
    '--root', str(ROBUST_ROOT),
    '--seeds', ','.join(map(str,cfg['seed_robustness']['seeds'])),
    '--context-seeds', ','.join(map(str,cfg['context_size']['seeds'])),
    '--context-sizes', ','.join(map(str,cfg['context_size']['train_sizes'])),
], check=True)
progress = json.loads((ROBUST_ROOT/'ROBUSTNESS_STATUS.json').read_text())
print({k:v.get('complete') for k,v in progress.items()})


## 6. Controlled Safe Selective CoRe — actual execution

This is a known-truth controlled mechanism experiment, not a fresh TFM benchmark. It saves every validation candidate score and applies structural-risk family selection before exact test scoring.


In [ ]:
SAFE_DIR = ROBUST_ROOT/'safe_selective'
if not (SAFE_DIR/'COMPLETE.json').exists():
    tmp = ROOT/'results'/'safe_selective_controlled_v1'
    if tmp.exists(): shutil.rmtree(tmp)
    subprocess.run([
        sys.executable, str(ROOT/'experiments'/'run_safe_selective_controlled.py'),
        '--tasks','100','--output-dir',str(tmp)
    ], cwd=ROOT, check=True)
    SAFE_DIR.mkdir(parents=True, exist_ok=True)
    for p in tmp.iterdir(): shutil.copy2(p, SAFE_DIR/p.name)
print(json.loads((SAFE_DIR/'COMPLETE.json').read_text()))


## 7. Archive-derived view-reliability diagnostic

The frozen Q1 table supports a raw-direction reliability proxy but does not contain every direct-marginal per-example score. This cell produces the strongest valid archive-derived diagnostic and labels it as a proxy. Fresh per-example view archives remain a separate enhancement and are not silently treated as completed evidence.


In [ ]:
VIEW_DIR = ROBUST_ROOT/'view_reliability'
VIEW_DIR.mkdir(parents=True, exist_ok=True)
derived = ROOT/'results'/'reliability_aware_v1'
subprocess.run([
    sys.executable, str(ROOT/'experiments'/'run_reliability_aware_suite.py'),
    '--fold-results', str(ROOT/'results'/'q1_fast_complete_256_v1'/'fold_results.csv'),
    '--output', str(derived)
], check=True)
for name in ['model_view_reliability_proxy.csv','inconsistency_vs_gain.csv','inconsistency_vs_gain_correlations.json']:
    shutil.copy2(derived/name, VIEW_DIR/name)
proxy_marker = {
    'complete':True, 'scope':'archive_derived_proxy_only',
    'fresh_per_example_direct_marginal_archive':False,
    'claim_guard':'May support diagnostic discussion; must not be described as completed fresh per-view TFM reruns.'
}
(VIEW_DIR/'PROXY_COMPLETE.json').write_text(json.dumps(proxy_marker, indent=2))
print(proxy_marker)


## 8. Final gate

`CORE ROBUSTNESS GATE` covers the experiments this notebook can truthfully complete now. `FULL JMLR GATE` additionally requires a fresh per-example four-view reliability archive and a successfully preflighted third TFM; those are not fabricated by this notebook.


In [ ]:
def exists(group, name='COMPLETE.json'):
    return (ROBUST_ROOT/group/name).exists()

core_status = {
    'multi_seed': exists('multi_seed'),
    'context_size': exists('context_size'),
    'rare_class_exclusion': exists('rare_class'),
    'safe_selective_controlled': exists('safe_selective'),
    'view_reliability_proxy': exists('view_reliability','PROXY_COMPLETE.json'),
}
full_jmlr_status = {
    **core_status,
    'fresh_per_example_view_reliability': exists('view_reliability','FRESH_COMPLETE.json'),
    'third_tfm': exists('third_tfm'),
}
print('CORE STATUS:', core_status)
print('CORE ROBUSTNESS GATE:', 'PASS' if all(core_status.values()) else 'PENDING')
print('FULL JMLR STATUS:', full_jmlr_status)
print('FULL JMLR GATE:', 'PASS' if all(full_jmlr_status.values()) else 'PENDING')
print('If real-TFM variants remain pending, rerun this notebook. It automatically resumes the next incomplete variant.')
